[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/mobilcom_course/blob/main/05_rural_macrocell_pathloss_interactive.ipynb)

> **Run this notebook in Google Colab** — click the badge above (no local install needed).
> The setup cell below installs dependencies and enables interactive `ipywidgets` sliders in Colab.


# 📡 Rural Macrocell (RMa) Path Loss — 3GPP TR 38.901
### Interactive Teaching Notebook — DHBW Mobile Communications
---
Implements the **3GPP Rural Macrocell** path-loss model (LOS & NLOS) from:
> 3GPP TR 38.901 — *Study on channel model for frequencies from 0.5 to 100 GHz* (Release 14).

Explore how carrier frequency, antenna heights, building height/street width, and the LOS/NLOS
condition shape the path loss over 2-D distance — including the **breakpoint** $d_{BP}$ where the
LOS slope changes.


In [ ]:
# ============================================================
#  COLAB / ENVIRONMENT SETUP  (safe to run locally too)
# ============================================================
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # numpy + matplotlib ship with Colab; ensure ipywidgets is present
    !pip install -q ipywidgets

    # Required so ipywidgets sliders / interactive output render in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
    print('✅ Colab detected — custom widget manager enabled.')
else:
    print('✅ Running locally (Jupyter) — no extra setup needed.')

# Inline backend works reliably for slider-driven redraws in both envs
%matplotlib inline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
print("✅ Libraries loaded.")


In [ ]:
# 3GPP RMa PATH-LOSS MODEL (TR 38.901)
c0 = 3e8  # speed of light [m/s]

def rma_pathloss(d_2D_m, fc_Hz, h_BS_m=35.0, h_UT_m=1.5, W_m=20.0, h_m=5.0):
    """
    Rural Macrocell path loss [dB] for a vector of 2-D distances.
    Returns (PL_LOS, PL_NLOS, d_BP_m).
        h_BS : base-station height [m]   (10..150)
        h_UT : user-terminal height [m]  (1..10)
        W    : street width [m]          (5..50)
        h    : avg. building height [m]  (5..50)
        fc   : carrier frequency [Hz]    (0.5..30 GHz for RMa)
    """
    d_2D = np.asarray(d_2D_m, dtype=float)
    fc_GHz = fc_Hz * 1e-9

    d_BP = 2*np.pi*h_BS_m*h_UT_m*fc_Hz/c0          # breakpoint distance [m]
    d_3D = np.sqrt((h_BS_m - h_UT_m)**2 + d_2D**2)  # 3-D distance
    d_3D_BP = np.sqrt((h_BS_m - h_UT_m)**2 + d_BP**2)

    # --- LOS ---
    def pl1(d3):
        return (20*np.log10(40*np.pi*d3*fc_GHz/3)
                + min(0.03*h_m**1.72, 10)*np.log10(d3)
                - min(0.044*h_m**1.72, 14.77)
                + 0.002*np.log10(h_m)*d3)

    PL1 = pl1(d_3D)
    PL2 = (40*np.log10(d_3D/d_BP) + pl1(d_3D_BP))
    PL_LOS = np.where((d_2D >= 10) & (d_2D <= d_BP), PL1, PL2)

    # --- NLOS ---
    PL_NLOS_prime = (161.04 - 7.11*np.log10(W_m) + 7.5*np.log10(h_m)
                     - (24.37 - 3.7*(h_m/h_BS_m)**2)*np.log10(h_BS_m)
                     + (43.42 - 3.1*np.log10(h_BS_m))*(np.log10(d_3D) - 3)
                     + 20*np.log10(fc_GHz)
                     - (3.2*(np.log10(11.75*h_UT_m))**2 - 4.97))
    PL_NLOS = np.maximum(PL_LOS, PL_NLOS_prime)

    return PL_LOS, PL_NLOS, d_BP

print("✅ RMa path-loss model defined.")


In [ ]:
# STATIC PLOT with MATLAB default parameters (LOS, fc = 26 GHz)
fc_Hz = 26e9
d_2D = np.linspace(10, 1e4, 1000)   # LOS: 10 m .. 10 km
PL_LOS, PL_NLOS, d_BP = rma_pathloss(d_2D, fc_Hz)

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#f5f5f5')
ax.set_facecolor('#eaf4fb')
ax.semilogx(d_2D, PL_LOS, color='#2980b9', lw=2.5, label='RMa LOS')
ax.axvline(d_BP, color='#f39c12', lw=2, linestyle='--',
           label=f'Breakpunkt d_BP = {d_BP:.0f} m')
ax.set_xlabel('2D distance [m]')
ax.set_ylabel('Path loss (Dämpfung) [dB]')
ax.set_title('Rural Macrocell, LOS  (fc = 26 GHz)', fontweight='bold')
ax.grid(True, which='both', linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# INTERACTIVE EXPLORER
style  = {'description_width': '170px'}
layout = widgets.Layout(width='460px')

sl_fc   = widgets.FloatSlider(value=26.0, min=0.5, max=30.0, step=0.5,
    description='Frequency fc [GHz]:', style=style, layout=layout, readout_format='.1f')
sl_hbs  = widgets.FloatSlider(value=35, min=10, max=150, step=1,
    description='h_BS [m]:', style=style, layout=layout, readout_format='.0f')
sl_hut  = widgets.FloatSlider(value=1.5, min=1, max=10, step=0.5,
    description='h_UT [m]:', style=style, layout=layout, readout_format='.1f')
sl_W    = widgets.FloatSlider(value=20, min=5, max=50, step=1,
    description='Street width W [m]:', style=style, layout=layout, readout_format='.0f')
sl_h    = widgets.FloatSlider(value=5, min=5, max=50, step=1,
    description='Building height h [m]:', style=style, layout=layout, readout_format='.0f')
dd_cond = widgets.Dropdown(options=['LOS', 'NLOS'], value='LOS',
    description='Condition:', style=style, layout=layout)

out = widgets.Output()

def update(_=None):
    with out:
        clear_output(wait=True)
        fc_Hz = sl_fc.value * 1e9
        los = (dd_cond.value == 'LOS')
        d_max = 1e4 if los else 5e3          # LOS: 10 m..10 km, NLOS: 10 m..5 km
        d_2D = np.linspace(10, d_max, 1000)
        PL_LOS, PL_NLOS, d_BP = rma_pathloss(d_2D, fc_Hz,
                                             h_BS_m=sl_hbs.value, h_UT_m=sl_hut.value,
                                             W_m=sl_W.value, h_m=sl_h.value)
        PL = PL_LOS if los else PL_NLOS
        color = '#2980b9' if los else '#e74c3c'

        fig, ax = plt.subplots(figsize=(11, 6))
        fig.patch.set_facecolor('#f5f5f5')
        ax.set_facecolor('#eaf4fb')
        ax.semilogx(d_2D, PL, color=color, lw=2.5, label=f'RMa {dd_cond.value}')
        if los and 10 < d_BP < d_max:
            ax.axvline(d_BP, color='#f39c12', lw=2, linestyle='--',
                       label=f'Breakpunkt d_BP = {d_BP:.0f} m')
        ax.set_xlabel('2D distance [m]')
        ax.set_ylabel('Path loss (Dämpfung) [dB]')
        ax.set_title(f'Rural Macrocell, {dd_cond.value}   |   fc = {sl_fc.value:.1f} GHz',
                     fontweight='bold')
        ax.grid(True, which='both', linestyle=':', alpha=0.5)
        ax.legend()
        plt.tight_layout()
        plt.show()

        print(f"  λ = {c0/fc_Hz*100:.2f} cm    |    Breakpunkt d_BP = {d_BP:.0f} m")
        print(f"  Path loss @ 100 m  = {np.interp(100,  d_2D, PL):6.1f} dB")
        print(f"  Path loss @ 1 km   = {np.interp(1000, d_2D, PL):6.1f} dB")
        if d_max >= 5000:
            print(f"  Path loss @ 5 km   = {np.interp(5000, d_2D, PL):6.1f} dB")

for w in [sl_fc, sl_hbs, sl_hut, sl_W, sl_h, dd_cond]:
    w.observe(update, names='value')

hint = ("<h3>⚙️ RMa Path-Loss Parameters</h3>"
        "<p style='color:#555;font-size:13px'>"
        "<b>W</b> (street width) and <b>h</b> (building height) only affect the <b>NLOS</b> curve.<br>"
        "<b>Breakpunkt</b> d_BP = 2π·h_BS·h_UT·fc/c₀ — LOS slope steepens beyond it.</p>")

ui = widgets.VBox([
    widgets.HTML(hint),
    widgets.HBox([widgets.VBox([sl_fc, sl_hbs, sl_hut]),
                  widgets.VBox([sl_W, sl_h, dd_cond])]),
    out
])
display(ui)
update()


## 📚 Theory Reference — 3GPP RMa Model (TR 38.901)

### LOS path loss (two segments, split at the breakpoint)

$$d_{BP} = \frac{2\pi \, h_{BS} \, h_{UT} \, f_c}{c_0}$$

For $10\,\text{m} \le d_{2D} \le d_{BP}$:

$$PL_1 = 20\log_{10}\!\left(\frac{40\pi d_{3D} f_c}{3}\right) + \min(0.03\,h^{1.72},\,10)\log_{10} d_{3D} - \min(0.044\,h^{1.72},\,14.77) + 0.002\log_{10}(h)\,d_{3D}$$

For $d_{BP} < d_{2D} \le 10\,\text{km}$:

$$PL_2 = 40\log_{10}\!\left(\frac{d_{3D}}{d_{BP}}\right) + PL_1\big(d_{3D,BP}\big)$$

> Note: $f_c$ is in **GHz** inside these formulas (per the TR 38.901 table note).

### NLOS path loss

$$PL_{NLOS} = \max\big(PL_{LOS},\; PL'_{NLOS}\big)$$

$$PL'_{NLOS} = 161.04 - 7.11\log_{10} W + 7.5\log_{10} h - \left(24.37 - 3.7\left(\tfrac{h}{h_{BS}}\right)^2\right)\log_{10} h_{BS} + \big(43.42 - 3.1\log_{10} h_{BS}\big)\big(\log_{10} d_{3D} - 3\big) + 20\log_{10} f_c - \big(3.2(\log_{10}(11.75\,h_{UT}))^2 - 4.97\big)$$

### Applicability ranges

| Parameter | Symbol | Range |
|-----------|--------|-------|
| BS height | $h_{BS}$ | 10 – 150 m |
| UT height | $h_{UT}$ | 1 – 10 m |
| Street width | $W$ | 5 – 50 m |
| Building height | $h$ | 5 – 50 m |
| Frequency | $f_c$ | 0.5 – 30 GHz |
| Distance (LOS) | $d_{2D}$ | 10 m – 10 km |
| Distance (NLOS) | $d_{2D}$ | 10 m – 5 km |
